# Calibration of the muon momenta

**Owner: B.** Measure the momentum scale of the muons in the two $\eta$ regions with the
J/ψ peak, and deliver it in `calibration.json` through a pull request. The code goes in
`calibration.py`; this notebook only calls it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import dimuon
import mc
from calibration import ETA_SPLIT, M_JPSI, Calibration, fit_peak

GROUP = 0           # TODO: your group number

## 1. The control sample

Muon pairs with a mass near the J/ψ. How many events? Which columns have missing values?

In [ ]:
control_raw = pd.read_hdf('data/bsmumu.h5', f'group{GROUP}/control')
print(len(control_raw), 'events')
control_raw.isna().sum()

## 2. Selection and mass

Needs `dimuon.select` (C) and `dimuon.invariant_mass` (A): pull `main` once they are
merged. Where is the peak? The J/ψ mass is 3.0969 GeV.

In [ ]:
control = dimuon.select(control_raw)
m = dimuon.invariant_mass(control)
print(f'{len(control)} events selected')

plt.hist(m, bins=np.linspace(2.85, 3.25, 81), histtype='step')
plt.axvline(M_JPSI, color='grey', lw=0.5)
plt.xlabel(r'$m_{\mu\mu}$ [GeV]');

## 3. First on the MC

The MC is perfectly calibrated: the fit must give scales close to 1. If not, the method is
wrong, not the detector.

In [ ]:
mc_control, _ = mc.generate('control', n_bkg=8000, seed=1)
Calibration().fit(dimuon.select(mc_control))

## 4. The scales of the data

Save them in `calibration.json`, and open a pull request with it: the analysis team reads
this file.

In [ ]:
cal = Calibration().fit(control)
cal.save()
cal

## 5. Check with the mixed events

Events with one muon in each region were **not** used in the fit. Before the correction,
where is their J/ψ peak? After `cal.apply`, is it at 3.0969 GeV? Plot both histograms.

In [ ]:
# TODO (B)